# Autoresearch Experiment Analysis

Analysis of autonomous hyperparameter tuning results from `results.tsv`.

This notebook is adapted for `autoresearch-mamba`. Unlike Karpathy's original analysis notebook, it works with the current 4-column Mamba TSV format:

- `commit`
- `val_bpb`
- `status`
- `description`

If a future run also logs `memory_gb`, the notebook will pick that up automatically.


In [1]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path


In [ ]:

results_path = Path("results.tsv")
if not results_path.exists():
    raise FileNotFoundError(f"Could not find {results_path.resolve()}")

df = pd.read_csv(results_path, sep="	")
df.columns = [str(c).strip() for c in df.columns]

required = {"commit", "val_bpb", "status", "description"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"results.tsv is missing required columns: {sorted(missing)}")

df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
df["status"] = df["status"].astype(str).str.strip().str.upper()
if "memory_gb" in df.columns:
    df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
else:
    df["memory_gb"] = np.nan

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)


In [ ]:

counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")


In [ ]:

kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    bpb = row["val_bpb"]
    desc = row["description"]
    if pd.notna(row["memory_gb"]):
        print(f"  #{i:3d}  bpb={bpb:.6f}  mem={row['memory_gb']:.1f}GB  {desc}")
    else:
        print(f"  #{i:3d}  bpb={bpb:.6f}  {desc}")


## Val BPB Over Time

Track how the best (kept) `val_bpb` evolves as experiments progress. The running minimum shows the current frontier.


In [ ]:

fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
if valid.empty:
    raise ValueError("No non-crash experiments available to plot.")

baseline_bpb = valid.loc[0, "val_bpb"]
kept_mask = valid["status"] == "KEEP"
if kept_mask.any():
    best_bpb = valid.loc[kept_mask, "val_bpb"].min()
else:
    best_bpb = valid["val_bpb"].min()

below = valid[valid["val_bpb"] <= baseline_bpb + 0.0005].copy()
if below.empty:
    below = valid.copy()

disc = below[below["status"] == "DISCARD"]
ax.scatter(
    disc.index,
    disc["val_bpb"],
    c="#cccccc",
    s=12,
    alpha=0.5,
    zorder=2,
    label="Discarded",
)

kept_v = below[below["status"] == "KEEP"]
ax.scatter(
    kept_v.index,
    kept_v["val_bpb"],
    c="#2ecc71",
    s=50,
    zorder=4,
    label="Kept",
    edgecolors="black",
    linewidths=0.5,
)

if kept_mask.any():
    kept_idx = valid.index[kept_mask]
    kept_bpb = valid.loc[kept_mask, "val_bpb"]
    running_min = kept_bpb.cummin()
    ax.step(
        kept_idx,
        running_min,
        where="post",
        color="#27ae60",
        linewidth=2,
        alpha=0.7,
        zorder=3,
        label="Running best",
    )

    for idx, bpb in zip(kept_idx, kept_bpb):
        desc = str(valid.loc[idx, "description"]).strip()
        if len(desc) > 45:
            desc = desc[:42] + "..."
        ax.annotate(
            desc,
            (idx, bpb),
            textcoords="offset points",
            xytext=(6, 6),
            fontsize=8.0,
            color="#1a7a3a",
            alpha=0.9,
            rotation=30,
            ha="left",
            va="bottom",
        )

n_total = len(df)
n_kept = int((df["status"] == "KEEP").sum())
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation BPB (lower is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

margin = max((baseline_bpb - best_bpb) * 0.15, 0.0005)
ax.set_ylim(best_bpb - margin, baseline_bpb + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")


## Summary Statistics


In [ ]:

kept = df[df["status"] == "KEEP"].copy()
baseline_bpb = df.iloc[0]["val_bpb"]
if kept.empty:
    best_bpb = df["val_bpb"].min()
    best_row = df.loc[df["val_bpb"].idxmin()]
else:
    best_bpb = kept["val_bpb"].min()
    best_row = kept.loc[kept["val_bpb"].idxmin()]

print(f"Baseline val_bpb:  {baseline_bpb:.6f}")
print(f"Best val_bpb:      {best_bpb:.6f}")
print(f"Total improvement: {baseline_bpb - best_bpb:.6f} ({(baseline_bpb - best_bpb) / baseline_bpb * 100:.2f}%)")
print(f"Best experiment:   {best_row['description']}")
print()

print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for _, row in kept_sorted.iterrows():
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: bpb={row['val_bpb']:.6f}  {desc}")


## Top Hits (Kept Experiments by Improvement)

Each kept experiment's delta is measured against the previous kept experiment, since experiments are cumulative and each one builds on the last kept state.


In [ ]:

kept = df[df["status"] == "KEEP"].copy()
kept["prev_bpb"] = kept["val_bpb"].shift(1)
kept["delta"] = kept["prev_bpb"] - kept["val_bpb"]

hits = kept.iloc[1:].copy()
hits = hits.sort_values("delta", ascending=False)

if hits.empty:
    print("No post-baseline kept improvements yet.")
else:
    print(f"{'Rank':>4}  {'Delta':>8}  {'BPB':>10}  Description")
    print("-" * 80)
    for rank, (_, row) in enumerate(hits.iterrows(), 1):
        print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_bpb']:.6f}  {row['description']}")

    print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")
